# TMS Fraud Detection

Reads orders, payments, order items, and customers from `SUMMIT_DB_DEV.TRANSFORM.DT_CLEAN_*` dynamic tables, scores each payment using the fraud detection heuristics, and writes results to `SUMMIT_DB_DEV.TRANSFORM.FRAUD_DETECTION_RESULTS`.

In [ ]:
import os
import snowflake.connector
from snowflake.snowpark.context import get_active_session
import pandas as pd
from datetime import datetime, timedelta, timezone

# If running locally, use the block below
conn = snowflake.connector.connect(
    connection_name="summit"
)

# If running in Snowsight Workspace, use the block below
# session = get_active_session()
# conn = session.connection

cur = conn.cursor()
cur.execute("USE ROLE SUMMIT_DEVELOPER_ROLE_DEV")
cur.execute("USE WAREHOUSE SUMMIT_WH_DEV")

In [ ]:
TRANSFORM_SCHEMA = "SUMMIT_DB_DEV.TRANSFORM"
OUTPUT_TABLE = "SUMMIT_DB_DEV.TRANSFORM.FRAUD_DETECTION_RESULTS"
FRAUD_THRESHOLD = 0.30

## Load data from Snowflake

In [ ]:
payments_df = cur.execute(f"SELECT * FROM {TRANSFORM_SCHEMA}.DT_CLEAN_PAYMENTS").fetch_pandas_all()
orders_df = cur.execute(f"SELECT * FROM {TRANSFORM_SCHEMA}.DT_CLEAN_ORDERS").fetch_pandas_all()
order_items_df = cur.execute(f"SELECT * FROM {TRANSFORM_SCHEMA}.DT_CLEAN_ORDER_ITEMS").fetch_pandas_all()
customers_df = cur.execute(f"SELECT * FROM {TRANSFORM_SCHEMA}.DT_CLEAN_CUSTOMERS").fetch_pandas_all()

print(f"Payments: {len(payments_df)}, Orders: {len(orders_df)}, Items: {len(order_items_df)}, Customers: {len(customers_df)}")

## Fraud detection logic

In [ ]:
COUNTRY_IP_PREFIXES = {
    "Germany": "91.23", "France": "82.45", "Netherlands": "145.90", "Czech Republic": "78.128",
    "Switzerland": "178.82", "Poland": "185.45", "Austria": "77.116", "Italy": "151.38",
    "Denmark": "212.88", "Spain": "88.12", "Belgium": "109.23", "Sweden": "83.55",
    "Ireland": "86.45", "Romania": "79.114", "Finland": "91.152", "Portugal": "95.92",
    "Hungary": "84.206", "Bulgaria": "78.83", "Greece": "94.66", "United Kingdom": "81.174",
}

FRAUD_DEVICE_POOL = ["fp_fraud_000", "fp_fraud_001", "fp_fraud_002", "fp_fraud_003", "fp_fraud_004"]

_recent_payments = []


def detect_fraud(payment, order, items, customer, threshold=FRAUD_THRESHOLD):
    signals = []
    score = 0.0

    billing_country = payment.get("BILLING_COUNTRY") or ""
    card_country = payment.get("CARD_COUNTRY") or ""
    customer_country = customer.get("COUNTRY") or ""

    if billing_country and billing_country != customer_country:
        signals.append("billing_country_mismatch")
        score += 0.35

    if card_country and card_country != customer_country:
        signals.append("card_country_mismatch")
        score += 0.30

    ip = payment.get("IP_ADDRESS") or ""
    ip_prefix = ".".join(ip.split(".")[:2]) if ip else ""
    expected_prefix = COUNTRY_IP_PREFIXES.get(customer_country, "")
    if ip_prefix and expected_prefix and ip_prefix != expected_prefix:
        signals.append("ip_geolocation_mismatch")
        score += 0.25

    fp = payment.get("DEVICE_FINGERPRINT") or ""
    if fp in FRAUD_DEVICE_POOL:
        signals.append("known_fraud_device")
        score += 0.40

    pay_ts = payment.get("PAYMENT_TIMESTAMP")
    if pay_ts:
        if isinstance(pay_ts, str):
            pay_ts = datetime.fromisoformat(pay_ts)
        cutoff = pay_ts - timedelta(minutes=5)
        recent_same_customer = [
            p for p in _recent_payments
            if p["customer_id"] == customer.get("CUSTOMER_ID")
            and p["timestamp"] > cutoff
        ]
        if len(recent_same_customer) >= 5:
            signals.append("velocity_abuse")
            score += 0.30

        _recent_payments.append({
            "customer_id": customer.get("CUSTOMER_ID"),
            "timestamp": pay_ts,
        })
        if len(_recent_payments) > 500:
            _recent_payments.pop(0)

    total_declared = sum(i.get("DECLARED_VALUE", 0) or 0 for i in items)
    if total_declared > 20000:
        signals.append("high_declared_value")
        score += 0.20

    return {
        "is_fraud": score >= threshold,
        "fraud_score": round(min(score, 1.0), 2),
        "signals": signals,
    }

## Score all payments

In [ ]:
customers_lookup = customers_df.set_index("CUSTOMER_ID").to_dict("index")
items_by_order = order_items_df.groupby("ORDER_ID").apply(
    lambda g: g.to_dict("records"), include_groups=False
).to_dict()

results = []
_recent_payments.clear()

for _, pay_row in payments_df.iterrows():
    payment = pay_row.to_dict()
    order_id = payment["ORDER_ID"]

    order_row = orders_df[orders_df["ORDER_ID"] == order_id]
    if order_row.empty:
        continue
    order = order_row.iloc[0].to_dict()

    customer_id = order.get("CUSTOMER_ID")
    customer = customers_lookup.get(customer_id, {})
    customer["CUSTOMER_ID"] = customer_id

    items = items_by_order.get(order_id, [])

    detection = detect_fraud(payment, order, items, customer)

    results.append({
        "PAYMENT_ID": payment["PAYMENT_ID"],
        "ORDER_ID": order_id,
        "CUSTOMER_ID": customer_id,
        "FRAUD_SCORE": detection["fraud_score"],
        "IS_FRAUD": detection["is_fraud"],
        "FRAUD_SIGNALS": ",".join(detection["signals"]) if detection["signals"] else None,
        "PAYMENT_AMOUNT": payment.get("PAYMENT_AMOUNT"),
        "PAYMENT_TIMESTAMP": payment.get("PAYMENT_TIMESTAMP"),
        "SCORED_AT": datetime.now(timezone.utc),
    })

results_df = pd.DataFrame(results)
print(f"Scored {len(results_df)} payments")
print(f"Flagged as fraud: {results_df['IS_FRAUD'].sum()}")
results_df.head(10)

## Write results to Snowflake

In [ ]:
from snowflake.connector.pandas_tools import write_pandas

db, schema, table = OUTPUT_TABLE.split(".")
cur.execute(f"TRUNCATE TABLE IF EXISTS {OUTPUT_TABLE}")

success, num_chunks, num_rows, _ = write_pandas(
    conn,
    results_df,
    table_name=table,
    database=db,
    schema=schema,
    overwrite=False,
    use_logical_type=True,
)

print(f"Wrote {num_rows} rows to {OUTPUT_TABLE}")

## AI Enrichment — Classify fraud type and generate explanation

Uses `AI_CLASSIFY` and `AI_COMPLETE` to enhance flagged payments with fraud type and natural language explanation.

In [ ]:
ai_enrich_query = f"""
SELECT
    fr.PAYMENT_ID,
    AI_CLASSIFY(
        CONCAT(
            'TRANSACTION SUMMARY: Payment of EUR ', fr.PAYMENT_AMOUNT::VARCHAR,
            ' with overall fraud score ', fr.FRAUD_SCORE::VARCHAR, '/1.0 (threshold 0.5). ',
            'DETECTED SIGNALS: ', COALESCE(fr.FRAUD_SIGNALS, 'none'), '. ',
            'SIGNAL SCORING BREAKDOWN: ',
            IFF(fr.FRAUD_SIGNALS LIKE '%billing_country_mismatch%',
                'billing_country_mismatch (+0.35): billing=' || COALESCE(pay.BILLING_COUNTRY, '?') || ' differs from customer=' || c.COUNTRY || '. ', ''),
            IFF(fr.FRAUD_SIGNALS LIKE '%card_country_mismatch%',
                'card_country_mismatch (+0.30): card issued in ' || COALESCE(pay.CARD_COUNTRY, '?') || ' but customer is in ' || c.COUNTRY || '. ', ''),
            IFF(fr.FRAUD_SIGNALS LIKE '%ip_geolocation_mismatch%',
                'ip_geolocation_mismatch (+0.25): IP ' || COALESCE(pay.IP_ADDRESS, '?') || ' does not match expected prefix for ' || c.COUNTRY || '. ', ''),
            IFF(fr.FRAUD_SIGNALS LIKE '%known_fraud_device%',
                'known_fraud_device (+0.40): device fingerprint ' || COALESCE(pay.DEVICE_FINGERPRINT, '?') || ' is in the known fraud device pool. ', ''),
            IFF(fr.FRAUD_SIGNALS LIKE '%velocity_abuse%',
                'velocity_abuse (+0.30): 5+ payments from same customer within 5 minutes. ', ''),
            IFF(fr.FRAUD_SIGNALS LIKE '%high_declared_value%',
                'high_declared_value (+0.20): total declared shipment value exceeds EUR 20000. ', ''),
            'CLASSIFICATION RULES: ',
            'identity_theft = geographic mismatches (billing or card country differs from customer country). ',
            'card_testing = known fraud device used, often with small amounts. ',
            'account_takeover = IP geolocation mismatch combined with other signals. ',
            'synthetic_identity = multiple signals including fraud device and geographic mismatches. ',
            'friendly_fraud = high declared value with minimal other signals.'
        ),
        ['identity_theft', 'card_testing', 'account_takeover', 'synthetic_identity', 'friendly_fraud'],
        {{'task_description': 'Classify the fraud type based on the SIGNAL SCORING BREAKDOWN and CLASSIFICATION RULES provided. Match the detected signals to the most appropriate fraud category.'}}
    ):labels[0]::VARCHAR AS FRAUD_TYPE,
    AI_COMPLETE(
        'mistral-large2',
        CONCAT(
            'You are a fraud analyst. In 2-3 sentences explain why this payment is fraudulent. ',
            'Signals: ', COALESCE(fr.FRAUD_SIGNALS, 'none'), '. Score: ', fr.FRAUD_SCORE::VARCHAR, '/1.0. ',
            'Amount: EUR ', fr.PAYMENT_AMOUNT::VARCHAR, '. ',
            'Customer: ', c.COUNTRY, '. Billing: ', COALESCE(pay.BILLING_COUNTRY, 'same'), '. ',
            'Card: ', COALESCE(pay.CARD_COUNTRY, 'N/A'), '. IP: ', COALESCE(pay.IP_ADDRESS, 'N/A'), '. ',
            'Device: ', COALESCE(pay.DEVICE_FINGERPRINT, 'N/A'), '.'
        )
    ) AS EXPLANATION
FROM {OUTPUT_TABLE} fr
JOIN {TRANSFORM_SCHEMA}.DT_CLEAN_PAYMENTS pay ON fr.PAYMENT_ID = pay.PAYMENT_ID
JOIN {TRANSFORM_SCHEMA}.DT_CLEAN_ORDERS o ON fr.ORDER_ID = o.ORDER_ID
JOIN {TRANSFORM_SCHEMA}.DT_CLEAN_CUSTOMERS c ON fr.CUSTOMER_ID = c.CUSTOMER_ID
WHERE fr.IS_FRAUD = TRUE
"""

ai_df = cur.execute(ai_enrich_query).fetch_pandas_all()
print(f"AI enriched {len(ai_df)} flagged payments")
ai_df.head()

In [ ]:
if not ai_df.empty:
    for _, row in ai_df.iterrows():
        fraud_type = str(row["FRAUD_TYPE"]).replace("'", "''") if row["FRAUD_TYPE"] else ""
        explanation = str(row["EXPLANATION"]).replace("'", "''") if row["EXPLANATION"] else ""
        payment_id = str(row["PAYMENT_ID"]).replace("'", "''")
        cur.execute(
            f"UPDATE {OUTPUT_TABLE} SET FRAUD_TYPE = '{fraud_type}', EXPLANATION = '{explanation}' WHERE PAYMENT_ID = '{payment_id}'"
        )
    print(f"Updated {len(ai_df)} rows with FRAUD_TYPE and EXPLANATION")
else:
    print("No fraudulent payments to enrich")

## Summary

In [ ]:
fraud_only = results_df[results_df["IS_FRAUD"] == True]
print(f"Total payments scored: {len(results_df)}")
print(f"Fraudulent:            {len(fraud_only)} ({len(fraud_only)/len(results_df)*100:.1f}%)")
print(f"\nSignal breakdown:")
all_signals = fraud_only["FRAUD_SIGNALS"].str.split(",").explode()
print(all_signals.value_counts().to_string())

if not ai_df.empty:
    print(f"\nAI Fraud Type breakdown:")
    print(ai_df["FRAUD_TYPE"].value_counts().to_string())
    print(f"\nSample AI explanations:")
    for _, row in ai_df.head(3).iterrows():
        print(f"  Payment {row['PAYMENT_ID']} [{row['FRAUD_TYPE']}]: {row['EXPLANATION'][:200]}")

conn.close()